# NPE tuning stage 4

In [1]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import itertools
import torch
import torch.nn as nn
from torch.distributions import Uniform
import sbi
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding
from sbi.inference import NPE_C
from sbi.diagnostics import run_sbc, check_sbc
import warnings
import sys
sys.path.append('../../pysimARG')
from discrete_uniform import DiscreteUniform
from LeaveLengthOut_NN import LeaveLengthOut_NN

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load genome data and clonal tree.

In [2]:
drop_col = range(16, 32)
theta_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")
x_test = np.delete(x_test, drop_col, axis=1)
print(theta_test.shape, x_test.shape)

nan_row_test = np.where(np.isnan(x_test) | np.isinf(x_test))[0]
print(nan_row_test)

theta_test = np.delete(theta_test, nan_row_test, axis=0)
theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = np.delete(x_test, nan_row_test, axis=0)
x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

print(theta_test.shape, x_test.shape)

(1000, 3) (1000, 30)
[865 899]
torch.Size([998, 3]) torch.Size([998, 30])


In [3]:
theta1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")
theta2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

x = np.vstack([x1, x2])
x = np.delete(x, drop_col, axis=1)
theta = np.vstack([theta1, theta2])
print(theta.shape, x.shape)

nan_row = np.where(np.isnan(x) | np.isinf(x))[0]
print(nan_row)

theta = np.delete(theta, nan_row, axis=0)
theta = torch.tensor(theta[:10000, :], device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = np.delete(x, nan_row, axis=0)
x = torch.tensor(x[:10000, :], device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

print(theta.shape, x.shape)

(20000, 3) (20000, 30)
[  114   681   706  1448  2554  2818  7211  7282  7329  7392  8938  9827
  9973 10223 10788 12192 13567 14388 14653]
torch.Size([10000, 3]) torch.Size([10000, 30])


## Test functions

In [4]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)
    
    return ks_results, p_values

In [5]:
def mahalanobis_error(theta_est_post, theta_test_numpy):
    maha_errors = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        samples = theta_est_post[i]
        truth = theta_test_numpy[i]
        post_mean = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)

        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, returning NaN.")
            continue
        
        diff = post_mean - truth
        maha_dist_sq = np.dot(np.dot(diff, inv_cov), diff)
        maha_errors[i] = np.sqrt(maha_dist_sq)
    return maha_errors

## Tuning setting

In [12]:
seeds = [1, 2, 3, 4, 5]
output_dim = [2, 4]
layers = [1, 2, 3, 4]
units = [32, 48, 64, 128, 256]
num_posterior_samples = 1000
param_array = np.array(list(itertools.product(output_dim, layers, units)))

In [13]:
stage4_p_values = np.full((param_array.shape[0], 3), np.nan)
stage4_D_stats = np.full((param_array.shape[0], 3), np.nan)
stage4_maha_errors = np.full((param_array.shape[0],), np.nan)
stage4_nll = np.full((param_array.shape[0],), np.nan)

## Baseline NPE

In [14]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))
prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [15]:
for k in range(param_array.shape[0]):
    num_outputs = param_array[k, 0]
    num_hidden_layers = param_array[k, 1]
    num_hiddens = param_array[k, 2]

    embedding_net = LeaveLengthOut_NN(
        input_dim=30,
        num_hiddens=num_hiddens,
        num_hidden_layers=num_hidden_layers,
        num_outputs=num_outputs)
    neural_posterior = posterior_nn(
        model="nsf",
        embedding_net=embedding_net
    )
    print(f"Iteration {k}")
    print(f"Running output dimension set {num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}...")

    seed = seeds[0]
    torch.manual_seed(seed)
    np.random.seed(seed)

    inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
    density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
        max_num_epochs=500
    )
    posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

    theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
    for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
        theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
                                            show_progress_bars=False, reject_outside_prior=False)
        theta_est_post[j, :, :] = theta_post.detach().numpy()

    parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
    theta_test_expanded = theta_test.unsqueeze(1)
    theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
    theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
    is_less_than_truth = theta_est_post_tensor < theta_test_expanded
    ranks = torch.sum(is_less_than_truth, dim=1)

    ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
    stage4_p_values[k, :] = p_values
    stage4_D_stats[k, :] = ks_results

    stage4_maha_errors[k] = np.mean(mahalanobis_error(theta_est_post, theta_test_numpy))

    lp = density_estimator_baseline.log_prob(theta_test, x_test)
    stage4_nll[k] = -lp.detach().cpu().mean().item()

Iteration 0
Running output dimension set 2, hidden layers 1, hidden units 32...
 Neural network successfully converged after 120 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 48.91it/s]


Iteration 1
Running output dimension set 2, hidden layers 1, hidden units 48...
 Neural network successfully converged after 69 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 48.67it/s]


Iteration 2
Running output dimension set 2, hidden layers 1, hidden units 64...
 Neural network successfully converged after 140 epochs.

...

Sampling posterior: 100%|██████████| 998/998 [00:21<00:00, 47.48it/s]


Iteration 38
Running output dimension set 4, hidden layers 4, hidden units 128...
 Neural network successfully converged after 101 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 47.98it/s]


Iteration 39
Running output dimension set 4, hidden layers 4, hidden units 256...
 Neural network successfully converged after 92 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 47.54it/s]


In [16]:
np.save('../../data/NPE_tuning/stage4_p_values.npy', stage4_p_values)
np.save('../../data/NPE_tuning/stage4_D_stats.npy', stage4_D_stats)
np.save('../../data/NPE_tuning/stage4_maha_errors.npy', stage4_maha_errors)
np.save('../../data/NPE_tuning/stage4_nll.npy', stage4_nll)

In [22]:
indices = np.argpartition(stage4_nll, 8)[:8]
print(indices)

[ 6 34 16 29 19  2  3 24]


In [26]:
stage4_p_values_final = np.full((len(seeds), 3, 8), np.nan)
stage4_D_stats_final = np.full((len(seeds), 3, 8), np.nan)
stage4_maha_errors_final = np.full((len(seeds), 8), np.nan)
stage4_nll_final = np.full((len(seeds), 8), np.nan)

In [27]:
for m in range(len(indices)):
    k = indices[m]
    num_outputs = param_array[k, 0]
    num_hidden_layers = param_array[k, 1]
    num_hiddens = param_array[k, 2]

    embedding_net = LeaveLengthOut_NN(
        input_dim=30,
        num_hiddens=num_hiddens,
        num_hidden_layers=num_hidden_layers,
        num_outputs=num_outputs)
    neural_posterior = posterior_nn(
        model="nsf",
        embedding_net=embedding_net
    )
    print(f"Iteration {m}")
    print(f"Running output dimension set {num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}...")
    
    for i in range(len(seeds)):
        print(f"Running seed {seeds[i]}...")
        seed = seeds[i]
        torch.manual_seed(seed)
        np.random.seed(seed)

        inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
        density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
            max_num_epochs=500
        )
        posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

        theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
        for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
            theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
                                                show_progress_bars=False, reject_outside_prior=False)
            theta_est_post[j, :, :] = theta_post.detach().numpy()

        parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
        theta_test_expanded = theta_test.unsqueeze(1)
        theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
        theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
        is_less_than_truth = theta_est_post_tensor < theta_test_expanded
        ranks = torch.sum(is_less_than_truth, dim=1)

        ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
        stage4_p_values_final[i, :, m] = p_values
        stage4_D_stats_final[i, :, m] = ks_results

        stage4_maha_errors_final[i, m] = np.mean(mahalanobis_error(theta_est_post, theta_test_numpy))

        lp = density_estimator_baseline.log_prob(theta_test, x_test)
        stage4_nll_final[i, m] = -lp.detach().cpu().mean().item()

Iteration 0
Running output dimension set 2, hidden layers 2, hidden units 48...
Running seed 1...
 Neural network successfully converged after 110 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:21<00:00, 47.23it/s]


Running seed 2...
 Neural network successfully converged after 31 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:20<00:00, 48.75it/s]


Running seed 3...
 Neural network successfully converged after 106 epochs.

...

Sampling posterior: 100%|██████████| 998/998 [00:21<00:00, 46.40it/s]


Running seed 4...
 Neural network successfully converged after 175 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:21<00:00, 46.60it/s]


Running seed 5...
 Neural network successfully converged after 100 epochs.

Sampling posterior: 100%|██████████| 998/998 [00:24<00:00, 41.29it/s]


In [28]:
np.save('../../data/NPE_tuning/stage4_p_values_final.npy', stage4_p_values_final)
np.save('../../data/NPE_tuning/stage4_D_stats_final.npy', stage4_D_stats_final)
np.save('../../data/NPE_tuning/stage4_maha_errors_final.npy', stage4_maha_errors_final)
np.save('../../data/NPE_tuning/stage4_nll_final.npy', stage4_nll_final)

In [30]:
print(np.mean(stage4_nll_final, axis=0))
print(np.median(stage4_nll_final, axis=0))

[-3.84284086 -3.38536868 -3.92661424 -3.61715131 -4.02349486 -3.89576187
 -4.31296868 -3.85864997]
[-3.93051505 -3.25876069 -4.05456495 -3.56263709 -4.23207855 -4.00055599
 -4.53391552 -3.89549303]


In [31]:
print(np.mean(stage4_maha_errors_final, axis=0))
print(np.median(stage4_maha_errors_final, axis=0))

[0.93446714 1.0990786  0.92485186 1.06658832 1.20027198 0.93392972
 1.04889687 1.06027127]
[0.89020891 1.09947806 0.92792236 1.07966398 1.25599646 0.8972454
 1.05305991 1.03362455]


In [34]:
print(np.argsort(np.mean(stage4_nll_final, axis=0)))
print(np.argsort(np.median(stage4_nll_final, axis=0)))

[6 4 2 5 7 0 3 1]
[6 4 2 5 0 7 3 1]


In [35]:
best_three_indices = [6, 4, 2]
best_three_in_all = indices[best_three_indices]
print(best_three_in_all)

[ 3 19 16]


In [36]:
print(param_array[best_three_in_all])

[[  2   1 128]
 [  2   4 256]
 [  2   4  48]]
